Final flow 
- all code in functions
- data pull from geoserver
- column descriptions and layer descriptions from csv
- style file from github
- output stored locally

1. add comments to functions
2. common functions between raster and vector 


1. fill in the functions
2. test for 1 layer

TODO: 
1. best id structure

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import os

import json
import xml.etree.ElementTree as ET
import datetime
# from datetime import datetime, timezone

import requests
from io import BytesIO

from rasterio.warp import transform_bounds

from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import mapping, box, Polygon

import sys
sys.path.append('..')
import constants

import pystac
from pystac.extensions.table import TableExtension
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension

In [2]:
GEOSERVER_BASE_URL = constants.GEOSERVER_BASE_URL
GEOSERVER_BASE_URL

'https://geoserver.core-stack.org:8443/geoserver'

Raster flow

In [3]:
def generate_raster_url(workspace,
                        layer_name,
                        geoserver_base_url,
                        output_format="geotiff"):
    wcs_url = (
        f"{geoserver_base_url}/{workspace}/wcs?"
        f"service=WCS&version=2.0.1&request=GetCoverage&"
        f"CoverageId={workspace}:{layer_name}&"
        f"format={output_format}"
    )
    # print("Raster URL:",wcs_url)
    return wcs_url

In [4]:
def read_raster_data(raster_url):

    #when reading from geoserver
    response = requests.get(raster_url, verify=False)
    response.raise_for_status()
    raster_data = BytesIO(response.content)

    #read the data and fetch the metadata
    with rasterio.open(raster_data) as r:
        crs = r.crs
        bounds = r.bounds
        bbox = [bounds.left, bounds.bottom, bounds.right, bounds.top]
        footprint = Polygon([
            [bounds.left, bounds.bottom],
            [bounds.left, bounds.top],
            [bounds.right, bounds.top],
            [bounds.right, bounds.bottom]
        ])
        data = r.read(1) #TODO: wouldn't work if there are multiple bands

        # id = os.path.basename(raster_url) #works when data is local
        # id = layer_name
        gsd = 10
        shape = r.shape
        data_type = str(r.dtypes[0])
        
        return (data,
                bbox,
                mapping(footprint),
                crs,
                # id,
                gsd,
                shape,
                data_type
                )

In [ ]:
def compute_ground_sample_distance():

In [48]:
def create_raster_item(raster_filepath,id):

    # raster_data,bbox,footprint,crs,id,gsd,shape,data_type = read_raster_data(raster_filepath)
    raster_data,bbox,footprint,crs,gsd,shape,data_type = read_raster_data(raster_filepath)

    raster_item = pystac.Item(id=id,
                        geometry=footprint,
                        bbox=bbox,
                        datetime=datetime.datetime.now(datetime.timezone.utc),
                        properties={
                            #   title
                            # description
                            # "gsd": gsd, #adding this in raster extension 
                        })
    
    #add certain metadata under projection extension
    proj_ext = ProjectionExtension.ext(raster_item, add_if_missing=True)
    proj_ext.epsg = crs
    proj_ext.shape = [shape[0], shape[1]]

    return raster_item

In [45]:
def add_raster_data_asset(raster_item,
                          geoserver_url
                          ):
    raster_item.add_asset("data", Asset(
    # href=os.path.join(data_url, os.path.relpath(raster_path, start=data_dir)), #TODO
    href=geoserver_url,
    roles=["data"],
    title="Raster Layer"))

    return raster_item

In [ ]:
def add_raster_extension(raster_item):
    raster_ext = RasterExtension.ext(raster_item.assets["data"], add_if_missing=True)
    raster_band = RasterBand.create(
        data_type=data_type, 
        spatial_resolution=gsd,
        # nodata=nodata
    )
    raster_ext.bands = [raster_band]    

In [55]:
def parse_raster_style_file(style_file_path):

    tree = ET.parse(style_file_path)
    root = tree.getroot()
    classes = []

    for entry in root.findall(".//paletteEntry"):
        class_info = {}
        for attr_key, attr_value in entry.attrib.items():
            if attr_key == "value":
                try:
                    class_info[attr_key] = int(attr_value)
                except ValueError:
                    class_info[attr_key] = attr_value
            else:
                class_info[attr_key] = attr_value
        classes.append(class_info)

    # If no paletteEntry tags are found, check for item tags
    if not classes:
        for entry in root.findall(".//item"):
            class_info = {}
            for attr_key, attr_value in entry.attrib.items():
                if attr_key == "value":
                    try:
                        class_info[attr_key] = int(attr_value)
                    except ValueError:
                        class_info[attr_key] = attr_value
                else:
                    class_info[attr_key] = attr_value
            classes.append(class_info)
    return classes

In [ ]:
def add_classification_extension():
    classification_ext = ClassificationExtension.ext(raster_item.assets["data"], add_if_missing=True)
    stac_classes = []
    for cls in style_info:
        stac_class_obj = Classification.create(
            value=int(cls["value"]),
            name=cls.get("label") or f"Class {cls['value']}",
            description=cls.get("label"),
            color_hint=cls['color'].replace('#','')
        )
        stac_classes.append(stac_class_obj)
    classification_ext.classes = stac_classes

In [ ]:
def add_stylefile_asset():
    raster_item.add_asset("style", Asset(
        href=os.path.join(data_url, os.path.relpath(raster_style_path, start=data_dir)),
        media_type=MediaType.XML,
        roles=["metadata"],
        title="Raster Style (QML)"
    ))

In [ ]:
def generate_raster_thumbnail(raster_data,
                              style_info,
                              output_path
                              ):
    
    unique_raster_values = np.unique(raster_data.compressed() if isinstance(raster_data, np.ma.MaskedArray) else raster_data)
    # Filter QML info to only include values present in the raster data
    filtered_style_info = [cls for cls in style_info if cls.get('value') in unique_raster_values]
    
    values = [cls['value'] for cls in filtered_style_info if 'value' in cls]
    colors = [cls['color'] for cls in filtered_style_info if 'color' in cls]
    
    # print(f"Parsed QML values: {values}")
    # print(f"Parsed QML colors: {colors}")
        
    try:
        if not values or not colors or len(values) != len(colors):
            raise ValueError("Invalid or insufficient palette information in QML file.")
    
        sorted_indices = np.argsort(values)
        sorted_values = np.array(values)[sorted_indices]
        sorted_colors = np.array(colors)[sorted_indices]

        cmap = ListedColormap(sorted_colors)
        bounds = np.array(sorted_values) - 0.5
        bounds = np.append(bounds, sorted_values[-1] + 0.5)
        norm = Normalize(vmin=bounds.min(), vmax=bounds.max())

    except ValueError as e:
        print(f"Skipping palette generation due to error: {e}. Using a default colormap.")
        cmap = 'gray'
        norm = None
    plt.figure(figsize=(3, 3), dpi=100)
    
    plt.imshow(raster_data, cmap=cmap, norm=norm, interpolation='none')
    plt.axis('off')

    #os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.close()

In [ ]:
def add_thumbnail_asset():
    raster_item.add_asset("thumbnail", Asset(
        href=os.path.join(data_url, os.path.relpath(raster_thumbnail_path, start=data_dir)),
        media_type=MediaType.PNG,
        roles=["thumbnail"],
        title="Raster Thumbnail (QML)"
    ))

In [ ]:
def read_layer_description(filepath,layer_name):

In [ ]:
def read_layer_mapping(filepath):

In [ ]:
def generate_raster_stac():
    read_raster_data()
    create_raster_item()
    add_raster_data_asset()
    add_raster_extension()
    parse_raster_style_file()
    add_classification_extension()
    generate_raster_thumbnail()
    add_stylefile_asset()
    add_thumbnail_asset()

Test raster flow for a layer

In [5]:
block_district_state_df = pd.DataFrame({
    'block' : ['gobindpur','mirzapur','koraput','badlapur'],
    'district' : ['saraikela-kharsawan','mirzapur','koraput','jaunpur'],
    'state' : ['jharkhand','uttar_pradesh','odisha','uttar_pradesh']
})

block_district_state_df

,block,district,state
0,gobindpur,saraikela-kharsawan,jharkhand
1,mirzapur,mirzapur,uttar_pradesh
2,koraput,koraput,odisha
3,badlapur,jaunpur,uttar_pradesh


In [6]:
block = 'badlapur'
district = block_district_state_df[block_district_state_df['block'] == block]['district'].iloc[0]
state = block_district_state_df[block_district_state_df['block'] == block]['state'].iloc[0]
print(state,district,block)

uttar_pradesh jaunpur badlapur


In [7]:
layer_mapping_df = pd.read_csv('../data/test_mapping.csv')

In [8]:
layer_name = 'land_use_land_cover_raster'

In [9]:
layer_mapping_df.columns

Index(['display name', 'layer_type', 'layer_name', 'ee_layer_name',
       'db_dataset_name', 'geoserver_workspace_name', 'start_year', 'end_year',
       'geoserver_layer_name', 'style_file_url'],
      dtype='object')

In [19]:
start_year = '19'
end_year = '20'

In [11]:
geoserver_workspace_name = layer_mapping_df[layer_mapping_df['layer_name'] == layer_name]['geoserver_workspace_name'].iloc[0]
geoserver_layer_name = layer_mapping_df[layer_mapping_df['layer_name'] == layer_name]['geoserver_layer_name'].iloc[0]
print(geoserver_workspace_name,geoserver_layer_name)

LULC_level_3 LULC_{start_year}_{end_year}_{block}_level_3


In [27]:
start_year

'19'

In [34]:
geoserver_layer_name = geoserver_layer_name.format(start_year = start_year,end_year = end_year, block = block)
geoserver_layer_name

'LULC_19_20_badlapur_level_3'

In [35]:
geoserver_url = generate_raster_url(
    workspace=geoserver_workspace_name,
    layer_name=geoserver_layer_name,
    geoserver_base_url=GEOSERVER_BASE_URL
)

In [44]:
'_'.join((layer_name, block,start_year,end_year))

'land_use_land_cover_raster_badlapur_19_20'

In [49]:
raster_item = create_raster_item(geoserver_url,
                                 id=geoserver_layer_name)

/home/nirzaree/miniconda3/envs/.stac/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'geoserver.core-stack.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [51]:
raster_item = add_raster_data_asset(raster_item,geoserver_url=geoserver_url)

In [54]:
layer_mapping_df.head()

,display name,layer_type,layer_name,ee_layer_name,db_dataset_name,geoserver_workspace_name,start_year,end_year,geoserver_layer_name,style_file_url
0,the name which will be displayed on the dashboard,raster or vector layer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Admin Boundaries,vector,admin_boundaries_vector,admin_boundary,Admin Boundary,panchayat_boundaries,NaN,NaN,{district}_{block},https://raw.githubusercontent.com/core-stack-o...
2,Aquifer,vector,aquifer_vector_vector,aquifer_vector,Aquifer,aquifer,NaN,NaN,aquifer_vector_{district}_{block},https://raw.githubusercontent.com/core-stack-o...
3,Drainage Lines,vector,drainage_lines_vector,drainage_lines,Drainage,drainage,NaN,NaN,{district}_{block},https://raw.githubusercontent.com/core-stack-o...
4,Surface Water Bodies,vector,swb_vector,swb2,Surface Water Bodies,swb,NaN,NaN,surface_waterbodies_{district}_{block},https://raw.githubusercontent.com/core-stack-o...


In [ ]:
parse_raster_style_file(style_file_path='../data/')